In [1]:
import numpy as np
import pandas as pd
import cv2


In [2]:
df_train = r"C:\Users\ASHWARY MAN\OneDrive\Desktop\Grape-Disease\train"
df_val = r"C:\Users\ASHWARY MAN\OneDrive\Desktop\Grape-Disease\val"

In [3]:
import os
def create_dataframe(path):

    image_paths = []
    labels = []

    for class_name in os.listdir(path):

        class_folder = os.path.join(path, class_name)

        if os.path.isdir(class_folder):

            for image in os.listdir(class_folder):

                image_paths.append(os.path.join(class_folder, image))
                labels.append(class_name)

    df = pd.DataFrame({
        "Image_Path": image_paths,
        "Label": labels
    })

    return df

In [4]:
train_df = create_dataframe(df_train)
val_df = create_dataframe(df_val)

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [6]:
train_df["Encoded"] = le.fit_transform(train_df["Label"])
val_df["Encoded"] = le.transform(val_df["Label"])

In [7]:
IMG_SIZE = 224

def load_images(df):

    images = []
    labels = []

    for _, row in df.iterrows():

        img = cv2.imread(row["Image_Path"])

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        images.append(img)

        labels.append(row["Encoded"])

    X = np.array(images, dtype=np.float32)

    y = np.array(labels)

    return X, y

In [8]:
X_train, y_train = load_images(train_df)
X_val, y_val = load_images(val_df)

In [9]:
#X_train = X_train/255.0
#X_val = X_val/255.0

In [10]:
import tensorflow as tf

In [11]:
base_model = tf.keras.applications.EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

In [12]:
base_model.trainable=False

In [13]:
inputs = tf.keras.layers.Input(shape=(224,224,3))

x = base_model(inputs, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(4, activation="softmax")(x)

model = tf.keras.models.Model(inputs, outputs)

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,054,695 (15.47 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [15]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [16]:
model.compile(optimizer=optimizer,loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [17]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)

In [18]:
history = model.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=20,batch_size=16,callbacks=[early_stop])

Epoch 1/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 64s 272ms/step - accuracy: 0.5239 - loss: 1.1508 - val_accuracy: 0.8027 - val_loss: 0.7785
Epoch 2/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 54s 267ms/step - accuracy: 0.8018 - loss: 0.6823 - val_accuracy: 0.8644 - val_loss: 0.5197
Epoch 3/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 292ms/step - accuracy: 0.8646 - loss: 0.5041 - val_accuracy: 0.9162 - val_loss: 0.3978
Epoch 4/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 290ms/step - accuracy: 0.8990 - loss: 0.3981 - val_accuracy: 0.9260 - val_loss: 0.3261
Epoch 5/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 66s 322ms/step - accuracy: 0.9191 - loss: 0.3312 - val_accuracy: 0.9359 - val_loss: 0.2805
Epoch 6/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 290ms/step - accuracy: 0.9274 - loss: 0.2867 - val_accuracy: 0.9408 - val_loss: 0.2480
Epoch 7/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 57s 279ms/step - accuracy: 0.9360 - loss: 0.2606 - val_accuracy: 0.9445 - val_loss: 0.2242
Epoch 8/20
204/204 ━━━━━━━━━━━━━━━━━━━━ 56s 272ms/step - accuracy: 0.9415 - loss: 0

In [19]:
X_train.min(),X_train.max()

(0.0, 255.0)

In [20]:
X_train.dtype,X_train[0].mean()

(dtype('float32'), 120.03165)

In [21]:
np.unique(y_train),np.bincount(y_train)

(array([0, 1, 2, 3]), array([ 942, 1107,  861,  339], dtype=int64))

In [22]:
from tensorflow.keras.metrics import Accuracy

In [23]:
loss, acc = model.evaluate(X_val,y_val,verbose=1)
loss,acc*100

26/26 ━━━━━━━━━━━━━━━━━━━━ 11s 419ms/step - accuracy: 0.9667 - loss: 0.1147


(0.11473115533590317, 96.67077660560608)

In [24]:
from sklearn.metrics import classification_report

pred = model.predict(X_val)
pred_classes = pred.argmax(axis=1)

print(classification_report(y_val, pred_classes, target_names=["Grape___Black_rot","Grape___Esca_(Black_Measles)","Grape___healthy","Grape___Leaf_blight_(Isariopsis_Leaf_Spot)"]))

26/26 ━━━━━━━━━━━━━━━━━━━━ 15s 505ms/step
                                            precision    recall  f1-score   support

                         Grape___Black_rot       0.95      0.93      0.94       236
              Grape___Esca_(Black_Measles)       0.94      0.96      0.95       276
                           Grape___healthy       1.00      1.00      1.00       215
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)       1.00      1.00      1.00        84

                                  accuracy                           0.97       811
                                 macro avg       0.97      0.97      0.97       811
                              weighted avg       0.97      0.97      0.97       811



In [25]:
base_model.trainable=True

In [26]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [27]:
optimizer2 = tf.keras.optimizers.Adam(1e-5)

model.compile(
    optimizer=optimizer2,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [28]:
history_fine = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 73s 299ms/step - accuracy: 0.9092 - loss: 0.3367 - val_accuracy: 0.9679 - val_loss: 0.1353
Epoch 2/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 58s 286ms/step - accuracy: 0.9483 - loss: 0.2093 - val_accuracy: 0.9753 - val_loss: 0.1203
Epoch 3/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 58s 286ms/step - accuracy: 0.9606 - loss: 0.1600 - val_accuracy: 0.9741 - val_loss: 0.1002
Epoch 4/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 58s 285ms/step - accuracy: 0.9720 - loss: 0.1234 - val_accuracy: 0.9753 - val_loss: 0.0861
Epoch 5/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 287ms/step - accuracy: 0.9754 - loss: 0.1052 - val_accuracy: 0.9753 - val_loss: 0.0754
Epoch 6/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 58s 286ms/step - accuracy: 0.9772 - loss: 0.0941 - val_accuracy: 0.9778 - val_loss: 0.0673
Epoch 7/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 288ms/step - accuracy: 0.9772 - loss: 0.0847 - val_accuracy: 0.9803 - val_loss: 0.0605
Epoch 8/10
204/204 ━━━━━━━━━━━━━━━━━━━━ 59s 288ms/step - accuracy: 0.9843 - loss: 0

In [29]:
from sklearn.metrics import classification_report

pred = model.predict(X_val)
pred_classes = pred.argmax(axis=1)

print(classification_report(y_val, pred_classes, target_names=["Grape___Black_rot","Grape___Esca_(Black_Measles)","Grape___healthy","Grape___Leaf_blight_(Isariopsis_Leaf_Spot)"]))

26/26 ━━━━━━━━━━━━━━━━━━━━ 16s 547ms/step
                                            precision    recall  f1-score   support

                         Grape___Black_rot       0.98      0.97      0.97       236
              Grape___Esca_(Black_Measles)       0.97      0.98      0.98       276
                           Grape___healthy       1.00      1.00      1.00       215
Grape___Leaf_blight_(Isariopsis_Leaf_Spot)       1.00      1.00      1.00        84

                                  accuracy                           0.98       811
                                 macro avg       0.99      0.99      0.99       811
                              weighted avg       0.98      0.98      0.98       811



In [30]:
model.save("model/efficientnetb0_grape.h5")